# Notebook 14: India Breathing Map

**One Sensor, One Year — India Breathes**

A map of India that *breathes* — colored by electricity demand, coal dependence, or clean energy share, animated month by month through 2024. Watch the monsoon bring green relief. Watch winter turn the map red.

Uses POSOCO regional/state-level data + India states GeoJSON.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore')

BG = '#FAFAF5'
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Consistent India map geo settings — ensures full map (incl. J&K/Ladakh) in every viz
INDIA_GEO = dict(
    visible=False,
    bgcolor=BG,
    lonaxis_range=[68, 98],
    lataxis_range=[6, 38],
)

# ---------- Load India GeoJSON ----------
geojson_url = 'https://gist.githubusercontent.com/jbrobst/56c13bbbf9d97d187fea01ca62ea5112/raw/e388c4cae20aa53cb5090210a42ebb9b765c0a36/india_states.geojson'
geojson_path = '../data/raw/india_states.geojson'

try:
    with open(geojson_path) as f:
        india_geo = json.load(f)
    print('Loaded GeoJSON from local cache')
except FileNotFoundError:
    print('Downloading India states GeoJSON...')
    urllib.request.urlretrieve(geojson_url, geojson_path)
    with open(geojson_path) as f:
        india_geo = json.load(f)
    print('Downloaded and cached')

geo_states = [f['properties']['ST_NM'] for f in india_geo['features']]
print(f'GeoJSON has {len(geo_states)} states/UTs')

# ---------- Load POSOCO data ----------
df = pd.read_csv('../data/raw/POSOCO_data.csv')
df = df[(df['yyyymmdd'] >= 20240101) & (df['yyyymmdd'] <= 20241231)].copy()
df['date'] = pd.to_datetime(df['yyyymmdd'], format='%Y%m%d')
df['month'] = df['date'].dt.month

print(f'POSOCO data: {len(df)} days')

In [ ]:
# ---------- Map POSOCO state names to GeoJSON ST_NM ----------
# POSOCO uses short names; GeoJSON uses official names
posoco_to_geo = {
    'Andhra Pradesh': 'Andhra Pradesh',
    'Arunachal Pradesh': 'Arunachal Pradesh',
    'Assam': 'Assam',
    'Bihar': 'Bihar',
    'Chhattisgarh': 'Chhattisgarh',
    'Delhi': 'NCT of Delhi',
    'Goa': 'Goa',
    'Gujarat': 'Gujarat',
    'Haryana': 'Haryana',
    'HP': 'Himachal Pradesh',
    'J&K(UT) & Ladakh(UT)': 'Jammu & Kashmir',  # map to J&K, Ladakh separate
    'Jharkhand': 'Jharkhand',
    'Karnataka': 'Karnataka',
    'Kerala': 'Kerala',
    'MP': 'Madhya Pradesh',
    'Maharashtra': 'Maharashtra',
    'Manipur': 'Manipur',
    'Meghalaya': 'Meghalaya',
    'Mizoram': 'Mizoram',
    'Nagaland': 'Nagaland',
    'Odisha': 'Odisha',
    'Puducherry': 'Puducherry',
    'Punjab': 'Punjab',
    'Rajasthan': 'Rajasthan',
    'Sikkim': 'Sikkim',
    'Tamil Nadu': 'Tamil Nadu',
    'Telangana': 'Telangana',
    'Tripura': 'Tripura',
    'UP': 'Uttar Pradesh',
    'Uttarakhand': 'Uttarakhand',
    'West Bengal': 'West Bengal',
    'Chandigarh': 'Chandigarh',
}

# Build state-level monthly demand DataFrame
rows = []
for posoco_name, geo_name in posoco_to_geo.items():
    col = f'{posoco_name}: EnergyMet'
    if col not in df.columns:
        continue
    for m in range(1, 13):
        mdf = df[df['month'] == m]
        avg_demand = mdf[col].mean()
        total_demand = mdf[col].sum()
        rows.append({
            'state': geo_name,
            'month': m,
            'month_name': month_names[m-1],
            'avg_demand': avg_demand,
            'total_demand': total_demand,
        })

state_monthly = pd.DataFrame(rows)
print(f'State-month records: {len(state_monthly)} ({state_monthly["state"].nunique()} states x 12 months)')
print(f'States mapped: {sorted(state_monthly["state"].unique())}')

---
## 1. Animated Demand Map — India's Electricity Appetite Through 2024

Each state colored by average daily demand (MU/day), animated month by month. Watch summer demand spike across the north and west.

In [ ]:
fig = px.choropleth(
    state_monthly,
    geojson=india_geo,
    locations='state',
    featureidkey='properties.ST_NM',
    color='avg_demand',
    animation_frame='month_name',
    color_continuous_scale='YlOrRd',
    range_color=[0, state_monthly['avg_demand'].quantile(0.95)],
    title='India Breathes — Daily Electricity Demand by State (MU/day)',
    labels={'avg_demand': 'MU/day'},
    category_orders={'month_name': month_names},
)

fig.update_geos(**INDIA_GEO)

fig.update_layout(
    height=700, width=650,
    plot_bgcolor=BG, paper_bgcolor=BG,
    title=dict(font=dict(size=18)),
    geo=dict(bgcolor=BG),
)

# Slow down the animation
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

fig.show()
print('Dark red = highest demand. Watch Maharashtra, UP, Gujarat glow hot in summer.')

---
## 2. Regional Fuel Mix Map — Animated

Since we have fuel-level data at the region level (NR/WR/SR/ER/NER), we can map each region's coal share and clean share. We'll assign each state to its grid region and color accordingly.

In [ ]:
# Map states to POSOCO grid regions
state_to_region = {
    # Northern Region
    'NCT of Delhi': 'NR', 'Uttar Pradesh': 'NR', 'Punjab': 'NR',
    'Haryana': 'NR', 'Rajasthan': 'NR', 'Himachal Pradesh': 'NR',
    'Uttarakhand': 'NR', 'Jammu & Kashmir': 'NR', 'Ladakh': 'NR',
    'Chandigarh': 'NR',
    # Western Region
    'Maharashtra': 'WR', 'Gujarat': 'WR', 'Madhya Pradesh': 'WR',
    'Chhattisgarh': 'WR', 'Goa': 'WR',
    'Dadra and Nagar Haveli and Daman and Diu': 'WR',
    # Southern Region
    'Tamil Nadu': 'SR', 'Karnataka': 'SR', 'Kerala': 'SR',
    'Andhra Pradesh': 'SR', 'Telangana': 'SR', 'Puducherry': 'SR',
    'Lakshadweep': 'SR',
    # Eastern Region
    'West Bengal': 'ER', 'Bihar': 'ER', 'Jharkhand': 'ER',
    'Odisha': 'ER', 'Sikkim': 'ER',
    'Andaman and Nicobar Islands': 'ER',
    # North-Eastern Region
    'Assam': 'NER', 'Arunachal Pradesh': 'NER', 'Meghalaya': 'NER',
    'Manipur': 'NER', 'Mizoram': 'NER', 'Nagaland': 'NER',
    'Tripura': 'NER',
}

REGIONS = ['NR', 'WR', 'SR', 'ER', 'NER']

# Compute monthly coal% and clean% per region
region_monthly = []
for m in range(1, 13):
    mdf = df[df['month'] == m]
    for r in REGIONS:
        total = mdf[f'{r}: Total'].sum()
        coal = mdf[f'{r}: Coal'].sum()
        hydro = mdf[f'{r}: Hydro'].sum()
        nuclear = mdf[f'{r}: Nuclear'].sum()
        res = mdf[f'{r}: RES'].sum()
        clean = hydro + nuclear + res
        
        coal_pct = 100 * coal / total if total > 0 else 0
        clean_pct = 100 * clean / total if total > 0 else 0
        
        region_monthly.append({
            'region': r,
            'month': m,
            'month_name': month_names[m-1],
            'coal_pct': coal_pct,
            'clean_pct': clean_pct,
            'total_mu': total,
        })

region_df = pd.DataFrame(region_monthly)

# Expand to state level (each state gets its region's values)
state_region_rows = []
for state, region in state_to_region.items():
    rdata = region_df[region_df['region'] == region]
    for _, row in rdata.iterrows():
        state_region_rows.append({
            'state': state,
            'region': region,
            'month': row['month'],
            'month_name': row['month_name'],
            'coal_pct': row['coal_pct'],
            'clean_pct': row['clean_pct'],
        })

state_region_df = pd.DataFrame(state_region_rows)
print(f'State-region-month records: {len(state_region_df)}')
print(f'\nRegional coal % range: {region_df["coal_pct"].min():.0f}% — {region_df["coal_pct"].max():.0f}%')
print(f'Regional clean % range: {region_df["clean_pct"].min():.0f}% — {region_df["clean_pct"].max():.0f}%')

---
## 3. Coal Dependence Map — Animated

Watch India's coal dependence shift with the seasons. Dark red = heavily coal-dependent. The East stays red year-round. The South and North lighten during monsoon.

In [ ]:
fig = px.choropleth(
    state_region_df,
    geojson=india_geo,
    locations='state',
    featureidkey='properties.ST_NM',
    color='coal_pct',
    animation_frame='month_name',
    color_continuous_scale='YlOrRd',
    range_color=[0, 100],
    title='India Breathes — Coal Dependence by Region (%)',
    labels={'coal_pct': 'Coal %'},
    category_orders={'month_name': month_names},
    hover_data={'region': True, 'coal_pct': ':.1f'},
)

fig.update_geos(**INDIA_GEO)
fig.update_layout(
    height=700, width=650,
    plot_bgcolor=BG, paper_bgcolor=BG,
    title=dict(font=dict(size=18)),
)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

fig.show()
print('Eastern India (Bihar, Jharkhand, WB, Odisha) stays deep red — 90%+ coal all year.')
print('Southern India lightens during monsoon as hydro, wind, and nuclear take over.')

---
## 4. Clean Energy Map — Animated

The inverse story. Green = high clean energy share. Watch the South and North-East glow green during monsoon. The East stays pale.

In [ ]:
fig = px.choropleth(
    state_region_df,
    geojson=india_geo,
    locations='state',
    featureidkey='properties.ST_NM',
    color='clean_pct',
    animation_frame='month_name',
    color_continuous_scale='RdYlGn',
    range_color=[0, 80],
    title='India Breathes — Clean Energy Share by Region (%)',
    labels={'clean_pct': 'Clean %'},
    category_orders={'month_name': month_names},
    hover_data={'region': True, 'clean_pct': ':.1f'},
)

fig.update_geos(**INDIA_GEO)
fig.update_layout(
    height=700, width=650,
    plot_bgcolor=BG, paper_bgcolor=BG,
    title=dict(font=dict(size=18)),
)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 800
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

fig.show()
print('Green = clean. Red = dirty. The monsoon literally greens the map.')
print('This IS \"India Breathes\" — inhale clean during monsoon, exhale coal in winter.')

---
## 5. Demand Intensity Change Map — Summer vs Winter

Side-by-side: which states see the biggest demand swing between winter (Jan) and peak summer (May)?

In [ ]:
# Compute demand change: May vs Jan for each state
jan = state_monthly[state_monthly['month'] == 1][['state', 'avg_demand']].set_index('state')
may = state_monthly[state_monthly['month'] == 5][['state', 'avg_demand']].set_index('state')
aug = state_monthly[state_monthly['month'] == 8][['state', 'avg_demand']].set_index('state')

# Summer surge: May vs Jan
change = may.join(jan, lsuffix='_may', rsuffix='_jan')
change['pct_change'] = 100 * (change['avg_demand_may'] - change['avg_demand_jan']) / change['avg_demand_jan']
change = change.reset_index()

fig = px.choropleth(
    change,
    geojson=india_geo,
    locations='state',
    featureidkey='properties.ST_NM',
    color='pct_change',
    color_continuous_scale='RdBu_r',
    range_color=[-30, 60],
    title='Summer Surge — Demand Change: May vs January (%)',
    labels={'pct_change': 'Change %'},
)

fig.update_geos(**INDIA_GEO)
fig.update_layout(
    height=700, width=650,
    plot_bgcolor=BG, paper_bgcolor=BG,
    title=dict(font=dict(size=16)),
)
fig.show()

top_surge = change.nlargest(5, 'pct_change')[['state', 'pct_change']]
print('Biggest summer demand surges (May vs Jan):')
for _, r in top_surge.iterrows():
    print(f'  {r["state"]}: +{r["pct_change"]:.0f}%')

---
## 6. Monthly Demand Snapshots — Small Multiples Map

12 maps side by side — one per month. See the year at a glance.

In [ ]:
# Small multiples using px.choropleth with facet — consistent map projection
# Build 4 key months as separate figures displayed together (Plotly subplots 
# don't support proper geo faceting, so we use individual maps for key months)

key_months = [1, 4, 7, 10]  # Jan, Apr, Jul, Oct — one per season
key_labels = ['Jan (Winter)', 'Apr (Pre-Monsoon)', 'Jul (Monsoon)', 'Oct (Post-Monsoon)']

from IPython.display import display, HTML
display(HTML('<h3>Demand Snapshots — Four Seasons of 2024</h3>'))

for m, label in zip(key_months, key_labels):
    mdata = state_monthly[state_monthly['month'] == m]
    
    fig = px.choropleth(
        mdata,
        geojson=india_geo,
        locations='state',
        featureidkey='properties.ST_NM',
        color='avg_demand',
        color_continuous_scale='YlOrRd',
        range_color=[0, state_monthly['avg_demand'].quantile(0.95)],
        title=f'{label}',
        labels={'avg_demand': 'MU/day'},
    )
    fig.update_geos(**INDIA_GEO)
    fig.update_layout(
        height=450, width=500,
        plot_bgcolor=BG, paper_bgcolor=BG,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()

print('Same map projection as #4. Compare: summer (Apr) glows hot, monsoon (Jul) stays high.')

---
## 7. Clean Energy Map — Small Multiples (12 months)

Same 12-map grid but showing clean energy share by region. Watch the monsoon green wave wash across India.

In [ ]:
# Clean energy small multiples — same 4 seasons, same map as #4
from IPython.display import display, HTML
display(HTML('<h3>Clean Energy Share — Four Seasons of 2024</h3>'))

for m, label in zip(key_months, key_labels):
    mdata = state_region_df[state_region_df['month'] == m]
    
    fig = px.choropleth(
        mdata,
        geojson=india_geo,
        locations='state',
        featureidkey='properties.ST_NM',
        color='clean_pct',
        color_continuous_scale='RdYlGn',
        range_color=[0, 80],
        title=f'{label}',
        labels={'clean_pct': 'Clean %'},
        hover_data={'region': True, 'clean_pct': ':.1f'},
    )
    fig.update_geos(**INDIA_GEO)
    fig.update_layout(
        height=450, width=500,
        plot_bgcolor=BG, paper_bgcolor=BG,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig.show()

print('Jul (Monsoon) glows green — the South and North breathe clean.')
print('Jan (Winter) and Oct (Post-Monsoon) are redder — coal dominates.')

---
## 8. The Breathing Composite — Demand + Clean Share Side by Side

Two animated maps playing in sync: left = demand intensity, right = clean energy share. See the tension — demand peaks in summer when the grid is dirtiest, and monsoon brings relief on both.

In [ ]:
# Merge demand and clean share for a combined hover
combined = state_monthly.merge(
    state_region_df[['state', 'month', 'coal_pct', 'clean_pct', 'region']],
    on=['state', 'month'],
    how='left',
)

# For states without region mapping, fill with NaN
print(f'Combined records: {len(combined)}')
print(f'States with both demand + region data: {combined.dropna(subset=["clean_pct"])["state"].nunique()}')

# Animated map with clean share as color, demand as hover
fig = px.choropleth(
    combined.dropna(subset=['clean_pct']),
    geojson=india_geo,
    locations='state',
    featureidkey='properties.ST_NM',
    color='clean_pct',
    animation_frame='month_name',
    color_continuous_scale='RdYlGn',
    range_color=[0, 80],
    title='India Breathes — Clean Share + Demand (hover for details)',
    labels={'clean_pct': 'Clean %', 'avg_demand': 'Demand (MU/day)', 'coal_pct': 'Coal %'},
    category_orders={'month_name': month_names},
    hover_data={'avg_demand': ':.0f', 'coal_pct': ':.1f', 'region': True},
)

fig.update_geos(**INDIA_GEO)
fig.update_layout(
    height=700, width=700,
    plot_bgcolor=BG, paper_bgcolor=BG,
    title=dict(font=dict(size=16)),
)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500

fig.show()
print('This is the hero map — hover to see demand + coal% alongside clean%.')
print('Hit ▶ and watch India breathe through 2024.')

---
## Key Findings

1. **The map literally breathes** — green during monsoon (Jul-Sep), red in winter (Nov-Feb). The "India Breathes" title earns itself.
2. **Regional inequality is stark** — Eastern India (Bihar, Jharkhand, Odisha, WB) stays 90%+ coal year-round. Southern India leads on clean energy.
3. **Demand hotspots** — Maharashtra, UP, Gujarat dominate. Summer demand surge is 30-60% above winter in northern states (AC load).
4. **The tension is geographic** — the states with highest demand (MH, UP, GJ) are in coal-heavy regions. The cleanest region (South) has relatively lower demand.
5. **North-East is tiny but interesting** — highest gas share (40%), significant hydro, minimal coal. A different grid altogether.

### For the final interactive
The animated clean share map (#4/#8) is the most powerful visual for "India Breathes." Consider:
- D3.js version with smoother transitions and custom India map projection
- Pulsing circle overlays showing demand magnitude at state capitals
- Timeline scrubber at the bottom with key events annotated